In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import re

import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import time
from datetime import timedelta
from tqdm import tqdm

import yfinance as yf
from gnews import GNews

import warnings
warnings.filterwarnings('ignore')

## Parameters

In [2]:
variables = []
with open("00 Variables.txt", "r") as f:
    lines = f.readlines()
    hashes = 1
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            hashes = len(s) - len(s.lstrip("#"))
            comment_indent = (hashes - 1) * 2
            print(" " * comment_indent + "\033[92m" + s + "\033[0m")
        elif "=" in s:
            var = s.split("=", 1)[0].strip()
            variables.append(var)
            try:
                exec(s)
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                value_part = s.split("#", 1)[0].strip()
                comment_part = s[s.find("#"):] if "#" in s else ""
                eval_value = f"{var} = {repr(eval(var))}"
                if comment_part:
                    print(" " * black_indent + eval_value + " \033[38;5;250m" + "  " + comment_part + "\033[0m")
                else:
                    print(" " * black_indent + eval_value)
            except Exception as e:
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                print(" " * black_indent + s)

# Global Settings
  TICKER = '^GSPC'
  START_DATE_GLOBAL = '1998-01-01'
  START_DATE = '2024-12-20'
  END_DATE = '2025-06-30'
  WINDOW_SIZE = 120
  HORIZON_DAYS = 1
  CRASH_PERCENTILE = 0.05
  MAX_WINDOWS = 10000
  DATE_INFO = True
  INDICATOR_SIZE = 'small'   # "small", "large"
  CHART_NUM = 4
  TRANSPARENT = False
  NEWS_QUERY = 'Standard and Poors 500 stock index'
  api_key = 'your-api-key'
  MODEL_NAME = 'gpt-4.1'   # "gpt-4.1", "o3-2025-04-16", "gpt-4.1-nano-2025-04-14"
  EMBEDDING_MODEL = 'text-embedding-3-small'
  RUN_TIMESTAMP = '20250726 1012'   # This will be changed automatically
  N_PREV = 6   # Number of previous windows to provide context for
  USE_IMAGES = True   # If True, send images (multimodal model), else just text
# Path
  ## Layer 1
     L1_RESULTS = 'Results'
     L1_COMB_RESULTS = 'Combination Results'
  ## Layer 2
     L2_DATA = 'Data'
     L2_FIGURES = 'Figures'
     L2_OUTPUT = 'Output'
  ## Layer 3
     L3_FIGURES_CHARTS = 'Generate Charts'
# Output Name
  O

## Definition

In [64]:
def download_and_prepare_prices(ticker, start, end):
    """
    Downloads historical stock prices for a given ticker and prepares the DataFrame.
    Args:
        - ticker: str, stock ticker symbol (e.g., 'AAPL')
        - start: str, start date in 'YYYY-MM-DD' format
        - end: str, end date in 'YYYY-MM-DD' format
    Returns:
        - df: pd.DataFrame, DataFrame with columns ['Close', 'High', 'Low', 'Open', 'Volume']
    """

    df_raw = yf.download(ticker, start=start, end=end)

    if isinstance(df_raw.columns, pd.MultiIndex):
        df_raw.columns = [c[0] for c in df_raw.columns]
        print("Flattened MultiIndex columns:", df_raw.columns.tolist())
    df = df_raw.copy()

    expected_cols = ['Close', 'High', 'Low', 'Open', 'Volume']
    missing = [col for col in expected_cols if col not in df.columns]
    if missing:
        print("Columns in data:", df.columns.tolist())
        raise ValueError(f"Missing columns in data: {missing}")

    df = df[expected_cols]

    for col in expected_cols:
        if isinstance(df[col].dtype, pd.StringDtype):
            print(f"Column {col} is a DataFrame, converting to Series...")
            df[col] = df[col].iloc[:, 0]
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    return df

In [65]:
def calculate_core_crash_indicators(df: pd.DataFrame):
    r = df.copy()
    r.columns = [str(col).lower() for col in r.columns]
    req = ['close', 'high', 'low', 'open', 'volume']
    for c in req:
        if c not in r.columns:
            raise ValueError(f"Missing required column: {c}")
        r[c] = pd.to_numeric(r[c], errors='coerce')
    if not isinstance(r.index, pd.DatetimeIndex):
        r.index = pd.to_datetime(r.index)
    r = r.sort_index()

    c, h, l, o, v = r['close'], r['high'], r['low'], r['open'], r['volume']
    eps = 1e-10

    # Bollinger Bands
    sma20 = c.rolling(20).mean()
    sma50 = c.rolling(50).mean()
    sma200 = c.rolling(200).mean()
    r['sma20'] = sma20
    r['sma50'] = sma50
    r['sma200'] = sma200

    std20 = c.rolling(20).std()
    r['bb_upper'] = sma20 + 2 * std20
    r['bb_lower'] = sma20 - 2 * std20
    r['bb_width'] = (r['bb_upper'] - r['bb_lower']) / (sma20 + eps)

    # SMA Crosses
    r['death_cross'] = ((sma50.shift(1) > sma200.shift(1)) & (sma50 < sma200)).astype(int)
    r['golden_cross'] = ((sma50.shift(1) < sma200.shift(1)) & (sma50 > sma200)).astype(int)

    # === Min/Max/Quantiles (for prompts and stats) ===
    r['minclose_20'] = c.rolling(20).min()
    r['maxclose_20'] = c.rolling(20).max()

    # Returns & Volatility
    r['logreturn'] = np.log(c / c.shift(1))
    n = WINDOW_SIZE
    r['volatility'] = r['logreturn'].rolling(n).std() * np.sqrt(252)
    r['hv_10'] = c.pct_change().rolling(10).std() * np.sqrt(252)
    r['hv_30'] = c.pct_change().rolling(30).std() * np.sqrt(252)
    r['vol_parkinson'] = (1.0 / (4.0 * np.log(2))) * ((np.log(h / l)) ** 2).rolling(n).mean().apply(np.sqrt) * np.sqrt(252)
    log_hl = np.log(h / l)
    log_co = np.log(c / o)
    r['vol_gk'] = 0.5 * (log_hl ** 2).rolling(n).mean() - (2 * np.log(2) - 1) * (log_co ** 2).rolling(n).mean()
    r['vol_gk'] = r['vol_gk'].apply(lambda x: np.sqrt(x) if x > 0 else np.nan) * np.sqrt(252)
    log_ho = np.log(h / o)
    log_lo = np.log(l / o)
    r['vol_rs'] = (log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)).rolling(n).mean().apply(np.sqrt) * np.sqrt(252)

    # === Composite Volatility ===
    ohlc_vols = ['volatility','vol_parkinson', 'vol_gk', 'vol_rs']
    r['composite_vol'] = r[ohlc_vols].mean(axis=1)
    r['volofvol'] = r['composite_vol'].rolling(n).std()

    for q in [0.01, 0.05, 0.95, 0.99]:
        r[f'r_{int(q*100)}pct'] = r['logreturn'].rolling(n).quantile(q)
        r[f'volatility_{int(q*100)}pct'] = r['volatility'].rolling(n).quantile(q)

    # MACD and histogram
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    r['macd'] = macd
    r['macd_hist'] = macd - macd_signal
    r['macd_signal'] = macd_signal

    # ROC
    r['roc_5'] = ((c / c.shift(5)) - 1) * 100
    r['roc_10'] = ((c / c.shift(10)) - 1) * 100
    r['roc_20'] = ((c / c.shift(20)) - 1) * 100

    # # Quantiles
    r['return_1pct'] = r['logreturn'].rolling(20).quantile(0.01)
    r['return_5pct'] = r['logreturn'].rolling(20).quantile(0.05)
    r['return_95pct'] = r['logreturn'].rolling(20).quantile(0.95)
    r['return_99pct'] = r['logreturn'].rolling(20).quantile(0.99)

    for q in [1, 5, 95, 99]:
        col = f'return_{q}pct'
        r[f'{col}_pct'] = (np.exp(r[col]) - 1) * 100

    # ATR
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    r['tr'] = tr
    r['atr'] = tr.rolling(14).mean()

    # === Chaikin Money Flow (CMF) ===
    mfm = ((c - l) - (h - c)) / (h - l + eps)
    r['cmf'] = (mfm * v).rolling(20).sum() / v.rolling(20).sum()

    # Stochastic Oscillator
    lowest_low = l.rolling(14).min()
    highest_high = h.rolling(14).max()
    r['stoch_k'] = 100 * (c - lowest_low) / (highest_high - lowest_low + eps)
    r['stoch_d'] = r['stoch_k'].rolling(3).mean()

    # Williams %R
    r['williamsr'] = -100 * ((highest_high - c) / (highest_high - lowest_low + eps))

    # Z-Score
    mean_price = c.rolling(20).mean()
    std_price = c.rolling(20).std()
    r['zscore'] = (c - mean_price) / (std_price + eps)
    
    # RSI (14)
    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss + eps)
    r['rsi'] = 100 - (100 / (1 + rs))

    # === CCI (Commodity Channel Index) ===
    tp = (h + l + c) / 3
    ma_tp = tp.rolling(20).mean()
    md = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    r['cci'] = (tp - ma_tp) / (0.015 * md + eps)

    # === Rolling Sharpe Ratio (20d) ===
    roll_mean = r['logreturn'].rolling(20).mean()
    roll_std = r['logreturn'].rolling(20).std()
    r['sharpe_20'] = roll_mean / (roll_std + eps) * np.sqrt(252)
    
    # === Ulcer Index (14d) ===
    max_close_14 = c.rolling(14).max()  # strict 14-day window
    drawdown = ((c - max_close_14) / max_close_14) * 100
    r['ulcer_index'] = np.sqrt((drawdown ** 2).rolling(14).mean())
    
    # === Rolling Maximum Drawdown (20d) ===
    roll_max_20 = c.rolling(20, min_periods=1).max()
    dd = (c / roll_max_20) - 1
    r['maxdd_20'] = dd.rolling(20, min_periods=1).min()

    # === True Range % (TR%) ===
    r['tr_percent'] = r['tr'] / c * 100

    # Tail risk: VaR/ES
    r['var_5pct_20'] = r['logreturn'].rolling(20).quantile(0.05)
    r['var_1pct_20'] = r['logreturn'].rolling(20).quantile(0.01)
    def es(x, p): return x.sort_values().iloc[:max(int(np.ceil(p * len(x))), 1)].mean() if len(x) else np.nan
    r['es_5pct_20'] = r['logreturn'].rolling(20).apply(lambda x: es(x, 0.05), raw=False)

    # OBV
    r['obv'] = (np.sign(c.diff()) * v).fillna(0).cumsum()

    # === ADX (Average Directional Index) ===
    plus_dm = h.diff().clip(lower=0)
    minus_dm = (-l.diff()).clip(lower=0)
    trur = r['tr'].rolling(14).sum()
    plus_di = 100 * plus_dm.rolling(14).sum() / (trur + eps)
    minus_di = 100 * minus_dm.rolling(14).sum() / (trur + eps)
    adx = (abs(plus_di - minus_di) / (plus_di + minus_di + eps)).rolling(14).mean() * 100
    r['adx'] = adx

    # === Statistical Moments ===
    r['skewness_20'] = r['logreturn'].rolling(20).skew()
    r['skewness_60'] = r['logreturn'].rolling(60).skew()
    r['kurtosis_20'] = r['logreturn'].rolling(20).kurt()
    r['kurtosis_60'] = r['logreturn'].rolling(60).kurt()

    return r

In [66]:
def compute_crash_labels_and_stats(price_df, prediction_dates, window_size=WINDOW_SIZE, horizon_days=HORIZON_DAYS, crash_percentile=CRASH_PERCENTILE, close_col='close'):
    close = price_df[close_col]
    logret = np.log(close).diff()
    quantile_series = (
        logret.rolling(window_size, min_periods=window_size)
        .quantile(crash_percentile)
        .shift(1)
    )
    labels, min_future_logrets, thresholds = [], [], []
    for d in tqdm(prediction_dates, desc="Crash label stats"):
        if d not in price_df.index or pd.isna(quantile_series.loc[d]):
            labels.append(np.nan)
            min_future_logrets.append(np.nan)
            thresholds.append(np.nan)
            continue
        idx = price_df.index.get_loc(d)
        crash_thr = quantile_series.iloc[idx]
        future_idx = np.arange(idx+1, min(idx+1+horizon_days, len(close)))
        if len(future_idx) == 0:
            labels.append(np.nan)
            min_future_logrets.append(np.nan)
            thresholds.append(crash_thr)
            continue
        rets = logret.iloc[future_idx]
        min_ret = rets.min()
        crash = int(min_ret <= crash_thr)
        labels.append(crash)
        min_future_logrets.append(min_ret)
        thresholds.append(crash_thr)
    return labels, min_future_logrets, thresholds


In [67]:
def fetch_and_cache_news(query, START_DATE_GLOBAL, end_date, cache_path: str, max_results=5, chunk_size=3):

    def remove_dates(text):
        date_patterns = [
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},\s+\d{4}',    # August 13, 2024 / Aug. 16, 2024
            r'\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}',     # 22-06-2012 or 22/06/2012
            r'\b\d{4}[-./]\d{1,2}[-./]\d{1,2}',     # # 2025.08.22 or 2025-08-22
            r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}',     # 22 August 2024
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}'     # August 2024
        ]
        combined_pattern = re.compile('|'.join(date_patterns), flags=re.IGNORECASE)
        if type(text)==str:
            value = combined_pattern.sub('', text)
        else:
            value = np.nan
        return value
    
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    gnews = GNews(language='en', country='US', max_results=max_results)
    all_news_list = []
    date_ranges = pd.date_range(START_DATE_GLOBAL, end_date, freq=f'{chunk_size}D')
    for chunk_start in tqdm(date_ranges, desc=f"Fetching {chunk_size}-day chunks"):
        chunk_end = min(chunk_start + pd.Timedelta(days=chunk_size-1), pd.Timestamp(end_date))
        after_str = chunk_start.strftime('%Y-%m-%d')
        before_str = (chunk_end + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        try:
            articles = gnews.get_news(f"{query} after:{after_str} before:{before_str}")
            news_text = " ".join([a.get('title', '') for a in articles])[:500]
        except Exception as e:
            print(f"[WARN] Error fetching news for {after_str} - {chunk_end}: {e}")
            news_text = ""
        for single_day in pd.date_range(chunk_start, chunk_end, freq='D'):
            all_news_list.append({"Date": single_day, "News": news_text})
        time.sleep(1)
        news_df = pd.DataFrame(all_news_list).set_index('Date')
        news_df = news_df[~news_df.index.duplicated()]
        news_df['News'] = news_df['News'].apply(remove_dates)
        news_df.to_csv(cache_path)
    return news_df

## Data Collection

In [68]:
print(f"\nDownloading price data for {TICKER} from {START_DATE_GLOBAL} to {END_DATE} ...")
df = download_and_prepare_prices(TICKER, START_DATE_GLOBAL, END_DATE)
print("Price data loaded:", df.shape)
display(df)

[*********************100%***********************]  1 of 1 completed


Flattened MultiIndex columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Price data loaded: (6914, 5)


,Close,High,Low,Open,Volume
Date,,,,,
1998-01-02,975.039978,975.039978,965.729980,970.429993,366730000
1998-01-05,977.070007,982.630005,969.000000,975.039978,628070000
1998-01-06,966.580017,977.070007,962.679993,977.070007,618360000
1998-01-07,964.000000,966.580017,952.669983,966.580017,667390000
1998-01-08,956.049988,964.000000,955.039978,964.000000,652140000
...,...,...,...,...,...
2025-06-23,6025.169922,6028.770020,5943.229980,5969.669922,5597000000
2025-06-24,6092.180176,6101.759766,6059.250000,6061.209961,5443690000
2025-06-25,6092.160156,6108.509766,6080.089844,6104.229980,5171110000


In [69]:
df['return'] = np.log(df['Close']).diff()   # or use pct_change if preferred
df = df.dropna().copy()

df_ind = calculate_core_crash_indicators(df)
df_ind_orig = df_ind.copy()
df_ind = df_ind.dropna().copy()

price_df = df_ind.copy()
price_df.index = pd.to_datetime(price_df.index)
price_df.columns = [c.lower() for c in price_df.columns]

logret = np.log(price_df['close']).diff()
quantile_series = (
    logret.rolling(WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .quantile(CRASH_PERCENTILE)
    .shift(1)
)
valid_dates = quantile_series.dropna().index

In [70]:
labels, min_future_logrets, thresholds = compute_crash_labels_and_stats(
    price_df, 
    price_df.index, 
    window_size=WINDOW_SIZE, 
    horizon_days=HORIZON_DAYS, 
    crash_percentile=CRASH_PERCENTILE,
    close_col='close'
)

df_labels = pd.DataFrame({
    "date": price_df.index,
    "crash_label": labels,
    "min_future_logret": min_future_logrets,
    "crash_threshold": thresholds
}).set_index('date')

Crash label stats: 100%|██████████| 6674/6674 [00:00<00:00, 25827.06it/s]


In [71]:
window_size=WINDOW_SIZE, 
horizon_days=HORIZON_DAYS, 
crash_percentile=CRASH_PERCENTILE,
close_col='close'

if close_col not in price_df.columns:
    lower_cols = [c.lower() for c in price_df.columns]
    if 'close' in lower_cols:
        close_col = price_df.columns[lower_cols.index('close')]
    else:
        raise ValueError(f"Column '{close_col}' not found in price_df.")

close = price_df[close_col]
logret = np.log(close).diff()
quantile_series = (
    logret.rolling(WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .quantile(CRASH_PERCENTILE)
    .shift(1)
)

In [72]:
df_full = pd.DataFrame(index=df.index.union(df_ind.index).union(df_labels.index).union(quantile_series.index))
df_full = df_full.assign(**df.to_dict('series'))
df_full = df_full.assign(**df_ind.to_dict('series'))
df_full = df_full.assign(**df_labels.to_dict('series'))
df_full['crash_quantile'] = quantile_series
df_full['logreturn'] = logret
df_full = df_full.sort_index()

In [73]:
indicators_list_small = [
    "close",
    "sma200",
    "bb_width",
    "death_cross",
    "volatility_1pct",
    "vol_gk",
    "atr",
    "macd_hist",
    "rsi",
    "williamsr",
    "ulcer_index",
    "var_5pct_20",
    "es_5pct_20",
    "return_99pct_pct",
    "return_1pct",
    "volume",
    "obv",
    "zscore",
    "skewness_60",
    "kurtosis_60"
]

df_ind_small = df_ind[indicators_list_small]

## Output

In [74]:
# 1. OHLCV
df_ohlcv_path = f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv"
os.makedirs(os.path.dirname(df_ohlcv_path), exist_ok=True)
df.to_csv(df_ohlcv_path, index=True, header=True)

# 2. Technical Indicators
df_indicators_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}.csv"
os.makedirs(os.path.dirname(df_indicators_path), exist_ok=True)
df_ind.to_csv(df_indicators_path, index=True, header=True)

# 3. Short Technical Indicators
df_ind_small = df_ind[indicators_list_small]
df_ind_small_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}_small.csv"
os.makedirs(os.path.dirname(df_ind_small_path), exist_ok=True)
df_ind_small.to_csv(df_ind_small_path, index=True, header=True)

# 4. Crash Labels & Stats (labels, min_future_logrets, thresholds)
df_crash_labels_path = f"{L1_RESULTS}/{L2_DATA}/{CRASH_LABEL}.csv"
os.makedirs(os.path.dirname(df_crash_labels_path), exist_ok=True)
df_labels = pd.DataFrame({
    "date": price_df.index,
    "crash_label": labels,
    "min_future_logret": min_future_logrets,
    "crash_threshold": thresholds
}).set_index("date")
df_labels.to_csv(df_crash_labels_path, index=True, header=True)

# 5. Rolling crash quantile/thresh series
df_quantile_series_path = f"{L1_RESULTS}/{L2_DATA}/{QUANTILE}.csv"
os.makedirs(os.path.dirname(df_quantile_series_path), exist_ok=True)
quantile_series.to_frame("crash_quantile").to_csv(df_quantile_series_path, index=True, header=True)

# 6. Log returns
df_logret_path = f"{L1_RESULTS}/{L2_DATA}/{LOG_RETURNS}.csv"
os.makedirs(os.path.dirname(df_logret_path), exist_ok=True)
logret.to_frame("logreturn").to_csv(df_logret_path, index=True, header=True)

# 7. Valid dates
df_valid_dates_path = f"{L1_RESULTS}/{L2_DATA}/{VALID_DATES}.csv"
os.makedirs(os.path.dirname(df_valid_dates_path), exist_ok=True)
pd.DataFrame({"date": valid_dates}).to_csv(df_valid_dates_path, index=False, header=True)

# 8. Full DataFrame
df_full_path = f"{L1_RESULTS}/{L2_DATA}/{FULL}.csv"
os.makedirs(os.path.dirname(df_full_path), exist_ok=True)
df_full.to_csv(df_full_path, index=True, header=True)

# 9. Full Indicators List
indicators_list_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS_LIST}.txt"
with open(indicators_list_path, "w") as f:
    for col in df_ind.columns:
        f.write(f"{col}\n")

# 10. Short Indicators List
indicators_list_small_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS_LIST}_small.txt"
os.makedirs(os.path.dirname(indicators_list_small_path), exist_ok=True)
with open(indicators_list_small_path, "w") as f:
    for col in indicators_list_small:
        f.write(f"{col}\n")

In [75]:
# NEWS_CACHE_PATH = f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv"

# print(f"Fetching news for {NEWS_QUERY} ...")
# news_df = fetch_and_cache_news(
#     NEWS_QUERY, START_DATE_GLOBAL, END_DATE, max_results=5, chunk_size=3, cache_path=NEWS_CACHE_PATH
# )
# print("News data loaded:", news_df.shape)

In [76]:
# NEWS_CACHE_PATH = f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv"

# print(f"Fetching news for {NEWS_QUERY} ...")
# news_df = fetch_and_cache_news(
#     NEWS_QUERY, START_DATE_GLOBAL, END_DATE, max_results=5, chunk_size=3, cache_path=NEWS_CACHE_PATH
# )
# print("News data loaded:", news_df.shape)

In [77]:
NEWS_CACHE_PATH = f"{NEWS_CACHE}.csv"
news_df = pd.read_csv(NEWS_CACHE_PATH, index_col='Date', parse_dates=True)
shutil.copy(NEWS_CACHE_PATH, f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv")

df_full = df_full.assign(**news_df.to_dict('series'))
df_full = df_full.sort_index()
df_full.to_csv(df_full_path, index=True, header=True)